# 01 — Eksplorasi Dataset LitSearch & Distribusi Panjang Token SPECTER2

Notebook ini menjalankan **Tahap 1 (Hari 1)** sesuai kerangka kerja penelitian:
1. Memuat dan memvalidasi integritas korpus resmi `princeton-nlp/LitSearch` (`64.183` artikel CS/AI, `597` kueri pencarian fleksibel, dan `qrels/test.tsv`).
2. Mengaudit kelengkapan metadata S2ORC (`title` dan `abstract`) serta distribusi atribut kueri (`specificity`, `quality`, `query_set`, dan jumlah *gold papers*).
3. Menghitung distribusi panjang token murni (`truncation=False`, `add_special_tokens=True`) menggunakan *tokenizer* resmi `allenai/specter2_base` untuk menyusun **TABLE I** naskah IEEE serta mengukur persentase dokumen yang melampaui batas `512` token.

In [1]:
import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    build_specter2_doc_text,
    load_litsearch_local,
    validate_litsearch_integrity,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'litsearch'
TABLES_DIR = PROJECT_ROOT / 'results' / 'tables'
FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

corpus, queries, qrels = load_litsearch_local(DATA_DIR)
integrity = validate_litsearch_integrity(corpus, queries, qrels)

both_present = sum(1 for d in corpus.values() if d['title'].strip() and d['text'].strip())
title_only = sum(1 for d in corpus.values() if d['title'].strip() and not d['text'].strip())
abstract_only = sum(1 for d in corpus.values() if not d['title'].strip() and d['text'].strip())
both_empty = sum(1 for d in corpus.values() if not d['title'].strip() and not d['text'].strip())

print('=== RINGKASAN INTEGRITAS LITSEARCH ===')
for k, v in integrity.items():
    print(f'  {k:20s}: {v}')
print('\n=== AUDIT KELENGKAPAN METADATA DOKUMEN (S2ORC) ===')
print(f'  Title + Abstract lengkap : {both_present} ({both_present / len(corpus) * 100:.2f}%)')
print(f'  Hanya Title tersedia     : {title_only} ({title_only / len(corpus) * 100:.2f}%)')
print(f'  Hanya Abstract tersedia  : {abstract_only} ({abstract_only / len(corpus) * 100:.2f}%)')
print(f'  Title & Abstract kosong  : {both_empty} ({both_empty / len(corpus) * 100:.2f}%)')

tokenizer = AutoTokenizer.from_pretrained('allenai/specter2_base')
tokenizer.model_max_length = 10_000_000
print(f'\nTokenizer berhasil dimuat: {tokenizer.__class__.__name__} | sep_token={tokenizer.sep_token!r}')

D:\tumgas\Kuliah\Semester 5\4. METODOLOGI PENELITIAN\project\metopen-specter2-litsearch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== RINGKASAN INTEGRITAS LITSEARCH ===
  corpus_count        : 64183
  query_count         : 597
  total_qrel_pairs    : 639
  missing_gold_count  : 0
  empty_titles        : 6760
  empty_abstracts     : 7989

=== AUDIT KELENGKAPAN METADATA DOKUMEN (S2ORC) ===
  Title + Abstract lengkap : 55631 (86.68%)
  Hanya Title tersedia     : 1792 (2.79%)
  Hanya Abstract tersedia  : 563 (0.88%)
  Title & Abstract kosong  : 6197 (9.66%)



Tokenizer berhasil dimuat: BertTokenizerFast | sep_token='[SEP]'


In [2]:
# Hitung panjang token murni (truncation=False, add_special_tokens=True)
query_ids = list(queries.keys())
query_texts = [queries[qid]['text'] for qid in query_ids]
q_enc = tokenizer(query_texts, truncation=False, add_special_tokens=True)
query_token_lens = np.array([len(ids) for ids in q_enc['input_ids']], dtype=np.int32)

doc_ids = list(corpus.keys())
doc_full_texts = [
    build_specter2_doc_text(corpus[did], sep_token=tokenizer.sep_token)
    for did in doc_ids
]
doc_titles = [(corpus[did]['title'] or '').strip() for did in doc_ids]
doc_abstracts = [(corpus[did]['text'] or '').strip() for did in doc_ids]

def batch_token_lengths(texts: list[str], batch_size: int = 2000, desc: str = 'Tokenizing') -> np.ndarray:
    lengths: list[int] = []
    for start in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[start : start + batch_size]
        enc = tokenizer(batch, truncation=False, add_special_tokens=True)
        lengths.extend(len(ids) for ids in enc['input_ids'])
    return np.array(lengths, dtype=np.int32)

doc_full_lens = batch_token_lengths(doc_full_texts, desc='Docs (Title+[SEP]+Abstract)')
doc_title_lens = batch_token_lengths(doc_titles, desc='Titles only')
doc_abs_lens = batch_token_lengths(doc_abstracts, desc='Abstracts only')

non_empty_mask = np.array([
    bool(corpus[did]['title'].strip() or corpus[did]['text'].strip())
    for did in doc_ids
])
doc_nonempty_lens = doc_full_lens[non_empty_mask]

print(f'Selesai tokenisasi! Total kueri={len(query_token_lens)}, Total dokumen={len(doc_full_lens)}')

Docs (Title+[SEP]+Abstract):   0%|          | 0/33 [00:00<?, ?it/s]

Docs (Title+[SEP]+Abstract):   3%|▎         | 1/33 [00:00<00:27,  1.15it/s]

Docs (Title+[SEP]+Abstract):   6%|▌         | 2/33 [00:01<00:21,  1.42it/s]

Docs (Title+[SEP]+Abstract):   9%|▉         | 3/33 [00:01<00:16,  1.80it/s]

Docs (Title+[SEP]+Abstract):  12%|█▏        | 4/33 [00:02<00:12,  2.29it/s]

Docs (Title+[SEP]+Abstract):  15%|█▌        | 5/33 [00:02<00:11,  2.53it/s]

... (72 outputs left out)

Abstracts only:  97%|█████████▋| 32/33 [00:18<00:00,  1.68it/s]

Abstracts only: 100%|██████████| 33/33 [00:18<00:00,  2.21it/s]

Abstracts only: 100%|██████████| 33/33 [00:18<00:00,  1.75it/s]

Selesai tokenisasi! Total kueri=597, Total dokumen=64183


In [3]:
# Analisis karakteristik kueri LitSearch (Gold papers, Specificity, Query Set, Quality)
df_queries = pd.DataFrame([
    {
        'query_id': q['_id'],
        'specificity': q['specificity'],
        'quality': q['quality'],
        'query_set': q['query_set'],
        'num_gold': len(q['gold_ids']),
        'token_len': int(tlen),
    }
    for q, tlen in zip(queries.values(), query_token_lens)
])

print('=== DISTRIBUSI JUMLAH GOLD PAPERS PER KUERI ===')
print(df_queries['num_gold'].describe().to_string())
print(f"Kueri dengan tepat 1 gold paper : {(df_queries['num_gold'] == 1).sum()} ({(df_queries['num_gold'] == 1).mean() * 100:.2f}%)")
print(f"Kueri dengan >= 2 gold papers   : {(df_queries['num_gold'] >= 2).sum()} ({(df_queries['num_gold'] >= 2).mean() * 100:.2f}%)")

breakdown_spec = (
    df_queries.groupby(['query_set', 'specificity'], as_index=False)
    .agg(
        num_queries=('query_id', 'count'),
        mean_tokens=('token_len', 'mean'),
        std_tokens=('token_len', 'std'),
        mean_gold=('num_gold', 'mean'),
    )
)
breakdown_spec['mean_tokens'] = breakdown_spec['mean_tokens'].map(lambda x: f'{x:.2f}')
breakdown_spec['std_tokens'] = breakdown_spec['std_tokens'].fillna(0.0).map(lambda x: f'{x:.2f}')
breakdown_spec['mean_gold'] = breakdown_spec['mean_gold'].map(lambda x: f'{x:.2f}')

breakdown_csv = TABLES_DIR / 'litsearch_query_breakdown.csv'
breakdown_spec.to_csv(breakdown_csv, index=False)
print(f'\n=== RINCIAN KATEGORI KUERI (Disimpan ke {breakdown_csv.name}) ===')
print(breakdown_spec.to_string(index=False))

=== DISTRIBUSI JUMLAH GOLD PAPERS PER KUERI ===
count    597.000000
mean       1.070352
std        0.325239
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        5.000000
Kueri dengan tepat 1 gold paper : 563 (94.30%)
Kueri dengan >= 2 gold papers   : 34 (5.70%)

=== RINCIAN KATEGORI KUERI (Disimpan ke litsearch_query_breakdown.csv) ===
    query_set  specificity  num_queries mean_tokens std_tokens mean_gold
   inline_acl            0           32       32.75      10.15      1.53
   inline_acl            1           66       33.23       8.78      1.18
inline_nonacl            0           88       24.73       5.40      1.09
inline_nonacl            1          165       26.81       5.55      1.02
   manual_acl            0           25       19.84       5.73      1.00
   manual_acl            1          130       24.02       6.80      1.00
  manual_iclr            0           10       24.30      12.30      1.10
  manual_iclr            1           81 

In [4]:
# Susun & ekspor TABLE I (results/tables/table1_dataset_stats.csv)
def summarize_tokens(component: str, arr: np.ndarray, max_seq_len: int = 512) -> dict:
    over_512 = int((arr > max_seq_len).sum())
    pct_over_512 = (over_512 / len(arr)) * 100.0
    return {
        'component': component,
        'count': int(len(arr)),
        'mean_tokens': f'{arr.mean():.2f}',
        'std_tokens': f'{arr.std(ddof=1):.2f}',
        'median_tokens': f'{np.median(arr):.2f}',
        'p95_tokens': f'{np.percentile(arr, 95):.2f}',
        'min_tokens': int(arr.min()),
        'max_tokens': int(arr.max()),
        'count_over_512': over_512,
        'pct_over_512': f'{pct_over_512:.2f}',
    }

broad_q_lens = df_queries.loc[df_queries['specificity'] == 0, 'token_len'].to_numpy(dtype=np.int32)
spec_q_lens = df_queries.loc[df_queries['specificity'] == 1, 'token_len'].to_numpy(dtype=np.int32)

table1_rows = [
    summarize_tokens('Queries (All Q_pos)', query_token_lens),
    summarize_tokens('Queries - Broad (specificity = 0)', broad_q_lens),
    summarize_tokens('Queries - Specific (specificity = 1)', spec_q_lens),
    summarize_tokens('Corpus Titles Only', doc_title_lens),
    summarize_tokens('Corpus Abstracts Only', doc_abs_lens),
    summarize_tokens('Corpus Full Input (Title + [SEP] + Abstract)', doc_full_lens),
    summarize_tokens('Corpus Non-Empty Docs Only (Diagnostic)', doc_nonempty_lens),
]

df_table1 = pd.DataFrame(table1_rows)
table1_csv = TABLES_DIR / 'table1_dataset_stats.csv'
df_table1.to_csv(table1_csv, index=False)

overview_json = {
    'integrity': integrity,
    'metadata_completeness': {
        'both_present': both_present,
        'title_only': title_only,
        'abstract_only': abstract_only,
        'both_empty': both_empty,
    },
    'query_summary': {
        'total_queries': int(len(df_queries)),
        'broad_specificity_0': int((df_queries['specificity'] == 0).sum()),
        'specific_specificity_1': int((df_queries['specificity'] == 1).sum()),
        'single_gold_queries': int((df_queries['num_gold'] == 1).sum()),
        'multi_gold_queries': int((df_queries['num_gold'] >= 2).sum()),
        'mean_gold_per_query': round(float(df_queries['num_gold'].mean()), 4),
        'max_gold_per_query': int(df_queries['num_gold'].max()),
        'unique_gold_papers': int(len({cid for g in qrels.values() for cid in g})),
    },
    'token_stats': table1_rows,
}
with (TABLES_DIR / 'dataset_overview_summary.json').open('w', encoding='utf-8') as f:
    json.dump(overview_json, f, indent=2)

print(f'=== TABLE I TERSIMPAN KE {table1_csv} ===')
print(df_table1.to_string(index=False))

=== TABLE I TERSIMPAN KE D:\tumgas\Kuliah\Semester 5\4. METODOLOGI PENELITIAN\project\metopen-specter2-litsearch\results\tables\table1_dataset_stats.csv ===


                                   component  count mean_tokens std_tokens median_tokens p95_tokens  min_tokens  max_tokens  count_over_512 pct_over_512
                         Queries (All Q_pos)    597       26.11       7.64         25.00      40.00          11          62               0         0.00
           Queries - Broad (specificity = 0)    155       25.57       8.23         25.00      37.00          12          62               0         0.00
        Queries - Specific (specificity = 1)    442       26.30       7.42         25.00      41.00          11          53               0         0.00
                          Corpus Titles Only  64183       14.28      30.45         13.00      25.00           2        3921              42         0.07
                       Corpus Abstracts Only  64183      162.67     130.79        153.00     327.00           2        4931            1239         1.93
Corpus Full Input (Title + [SEP] + Abstract)  64183      175.95     135.69       

In [5]:
# Visualisasi eksplorasi distribusi panjang token dokumen sesuai aturan ketat IEEE
# (lebar 1 kolom 3.5 in, 300 DPI, Times New Roman 8 pt, tanpa plt.title())
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
})

fig, ax = plt.subplots(figsize=(3.5, 2.4), dpi=300)
clipped_lens = np.clip(doc_full_lens, 0, 800)
pct_over = (doc_full_lens > 512).mean() * 100
ax.hist(
    clipped_lens,
    bins=45,
    color='#2b5c8f',
    edgecolor='white',
    linewidth=0.35,
    alpha=0.92,
    label='Corpus docs (clipped at 800)',
)
ax.vlines(
    512,
    0,
    6300,
    colors='#c0392b',
    linestyles='--',
    linewidth=1.15,
    label=f'Max seq. length = 512 ({pct_over:.2f}% > 512)',
)
ax.annotate(
    f'512 tokens\n({pct_over:.2f}% > 512)',
    xy=(512, 4300),
    xytext=(548, 4300),
    fontsize=6.5,
    color='#922b21',
    va='center',
    arrowprops={'arrowstyle': '->', 'color': '#c0392b', 'lw': 0.75},
)
ax.set_xlabel('SPECTER2 Token Length (Tokens)')
ax.set_ylabel('Number of Documents')
ax.set_xlim(0, 800)
ax.set_ylim(0, 8800)
ax.grid(axis='y', linestyle=':', linewidth=0.5, alpha=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper right', frameon=True, framealpha=1.0, edgecolor='#cccccc')
fig.tight_layout()

fig_png = FIGURES_DIR / 'eda_token_length_dist.png'
fig_pdf = FIGURES_DIR / 'eda_token_length_dist.pdf'
fig.savefig(fig_png, dpi=300, bbox_inches='tight')
fig.savefig(fig_pdf, bbox_inches='tight')
plt.close(fig)
print(f'[OK] Grafik distribusi token tersimpan: {fig_png.name} & {fig_pdf.name}')

[OK] Grafik distribusi token tersimpan: eda_token_length_dist.png & eda_token_length_dist.pdf
